In [1]:
import numpy as np

In [ ]:
class Neural:
  def __init__(self,input_size=784,hidden_layers=[64,64],output_size=10):
    self.input_size = input_size
    self.hidden_layers = hidden_layers
    self.output_size = output_size
    self.weights = []
    self.biases = []
    self.layers = [] # Post-ReLU values
    self.layers_z = [] #Pre-ReLU values

    #Input for hidden layers
    self.weights.append(0.01 * np.random.randn(input_size,hidden_layers[0]))
    self.biases.append(np.zeros((1,hidden_layers[0])))

    #Hidden Layers Network
    for i in range(len(hidden_layers)-1):
      self.weights.append(0.01 * np.random.randn(hidden_layers[i],hidden_layers[i+1]))
      self.biases.append(np.zeros((1,hidden_layers[i+1])))

    #Hidden layers to Output
    self.weights.append(0.01 * np.random.randn(hidden_layers[-1],output_size))
    self.biases.append(np.zeros((1,output_size)))

  def forward(self,inputs):
    self.layers = [inputs]
    self.layers_z = []
    #Dot product of inputs and weights
    for i in range(len(self.weights)):
      if i == (len(self.weights)-1):
        self.layers_z.append(np.dot(self.layers[-1],self.weights[i])+self.biases[i])
      else: 
        output = np.dot(self.layers[-1],self.weights[i])+self.biases[i]
        self.layers_z.append(output)
        output = self.relu(output)
        self.layers.append(output)
    self.layers.append(self.softmax_act(self.layers_z[-1]))
    return self.layers[-1]

  def relu(self,inputs):
    return np.where(inputs < 0, 0, inputs)

  def softmax_act(self,inputs):
    exp_values = np.exp(inputs)
    return exp_values / np.sum(exp_values,axis=1,keepdims=True)

  def loss(self,input,target,epsilon=1e-15):
    # Clip predictions to avoid numerical instability (log(0))
    y_pred = np.clip(input, epsilon, 1 - epsilon)
    
    # Calculate loss for each sample and then the average over the batch
    loss = -np.mean(np.sum(target * np.log(y_pred), axis=-1))
    return loss


In [3]:
neural_net = Neural()

In [4]:
img_data = np.load("Datasets/train_images.npy")
img_labels = np.load("Datasets/train_labels.npy")

In [5]:
img_data = np.reshape(img_data,(60000,784))
img_labels = np.reshape(img_labels,(60000,1))

In [6]:
img_norm = img_data/255

In [7]:
def true_vals(array):
  array = array.flatten()
  true_values = np.zeros((array.size,array.max()+1))
  true_values[np.arange(array.size),array] = 1
  return true_values

In [8]:
labels = true_vals(img_labels)

In [9]:
output = neural_net.forward(inputs=img_norm)
output

array([[0.09996999, 0.09997713, 0.09997858, ..., 0.09999756, 0.09997753,
        0.09999703],
       [0.09996307, 0.0999559 , 0.10000604, ..., 0.10000871, 0.09999558,
        0.09998335],
       [0.10000835, 0.09992648, 0.1000174 , ..., 0.09997717, 0.09998282,
        0.10004947],
       ...,
       [0.09995835, 0.09996546, 0.10002113, ..., 0.1000044 , 0.09996482,
        0.09999966],
       [0.09997695, 0.09997085, 0.10001002, ..., 0.0999878 , 0.09999835,
        0.0999797 ],
       [0.09999523, 0.09996737, 0.10001005, ..., 0.09999404, 0.09998881,
        0.09996453]], shape=(60000, 10))

In [61]:
loss = neural_net.loss(output,labels)
loss

np.float64(2.302615193300977)